# Étape 2

Étape 2 : Étape de nettoyage Dans cette étape, on s’intéresse à implémenter les correctifs soulignés dans l’étape 1. De ce fait, il serait important de considérer les opérations suivantes : 

* Imputing: évaluer les colonnes avec des valeurs manquantes. Par exemple, voir les colonnes: 
* title, revol_util et pub_rec_bankruptcies Une stratégie à employer: Supprimer la colonne ayant plus de 1 à 2% de valeurs manquantes Supprimer les lignes ayant des valeurs NaN  

* Convertir les colonnes catégorielles en numériques. Faire attention ici aux valeurs ordinales et nominales (dummy var).  

* Suppression de colonnes non adéquates pour la prédiction  

* Suppression des colonnes qui ne seront intéressantes que pour prédire le statut de paiement du prêt. Les colonnes dont les valeurs seront obtenues après le prêt ne doivent pas être considérées. Par exemple, évaluer si les colonnes suivantes sont à supprimer:  

	* zip_code  
	* out_prncp  
	* out_prncp_inv  
	* total_pymnt  
	* total_pymnt_inv  
	* total_rec_prncp  
	* total_rec_int  
	* total_rec_late_fee  
	* recoveries  
	* collection_recovery_fee  
	* last_pymnt_d  
	* last_pymnt_amnt  

Vérifier si la colonne cible est dans un format adéquat pour le modèle.  

Correction/Standardisation/Normalisation de données  

In [68]:

import pandas as pd 
import numpy as np
from utils.utils import distributional_summary, degree_completeness, degree_validity, get_serie_type, get_df_types, SerieValidityMapper
import matplotlib.pyplot as plt 
from typing import Dict, List, Callable, Any, Tuple


df_data = pd.read_csv('data/lending_club_loans.csv') 
labels = pd.read_excel('data/lending_club_data_dic.xlsx') 
labels.index = labels['LoanStatNew']
labels = labels.drop(columns=['LoanStatNew'])
labels = labels.loc[df_data.columns] # ! limits labels to our variables of interests


def convert_percentage(serie:pd.Series):
  return [ float(str(v).replace('%', '')) for v in serie.values]

def convert_date(serie:pd.Series): 
  return pd.to_datetime(serie, format='%b-%y') 

def delta_month(d1:pd.Series, d2:pd.Series):
	return ( (d1.dt.year - d2.dt.year) * 12 + (d1.dt.month - d2.dt.month) ).astype('Int64')

def mean_category(serie:pd.Series, encoding:dict) -> str:
	reversed = { v:k for k,v in encoding.items() } # ! reverse to encoding to get conversion FROM cat -> num instead of num -> cat 
	mean = round(pd.Series([ reversed.get(v, np.nan) for v in serie.values ]).mean(), 0) 
	return encoding[mean] 



# ! Identify varnames groups for particular treatments later 
summary = distributional_summary(df_data) 
varname_id = ['id', 'member_id'] 
varname_cardinality_1 = summary.loc[:, summary.loc['cardinality'] == 1].columns # ! to document. 
varname_date = ['last_pymnt_d', 'last_credit_pull_d', 'earliest_cr_line', 'issue_d'] 
varname_percentage = ['int_rate', 'revol_util'] 


## Encodage des variables categoriques

In [69]:

# ! Infer encoding values for categorical values 
df = df_data.copy() 
cat_var = ['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'addr_state'] 

encodings = {
  'emp_length':{
    0: '< 1 year',
    1: '1 year',
    2: '2 years',
    3: '3 years',
    4: '4 years',
    5: '5 years',
    6: '6 years',
    7: '7 years',
    8: '8 years',
    9: '9 years',
    10: '10+ years'}
}
for c in cat_var:
	encodings[c] = { i:v for i,v in enumerate(sorted(df[c].unique()))}


### Regle de transformations. 

Variables a retirer 
  'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', # ! There
  'last_pymnt_d', 'last_pymnt_amnt'
] 

In [70]:

# ? Template for transformation function. Helps intellisense and structures
type TransformerFunc = Callable[[pd.DataFrame, str], pd.Series] 

transform_mapper:Dict[str, Tuple[TransformerFunc, str]] = { 
	# ! convert string percentage -> numerical percentage 
  'int_rate': (lambda df,c: convert_percentage(df[c]), None), 
  'revol_util' : (lambda df,c: convert_percentage(df[c]), None), 
  
  # ! convert string date -> datetime values 
  'last_pymnt_d':	(lambda df,c: convert_date(df[c]), None), 
  'last_credit_pull_d': (lambda df,c: convert_date(df[c]), None), 
  'earliest_cr_line': (lambda df,c: convert_date(df[c]), None), 
  'issue_d': (lambda df,c: convert_date(df[c]), None), 
} 

imputations_mapper:Dict[str, Tuple[TransformerFunc, str]] = { 
	# ! Impute mean category from ORDINAL variable "emp_length" instead of using most frequent value. 
  'emp_length': (lambda df,c: mean_category(df[c], encodings[c]), 'emp_length'), 
	
	# ! Impute mean and round to integer value
	'revol_util': (lambda df, c: int(round(df[c].mean(), 0)), 'revol_util'), 
	'pub_rec_bankruptcies': (lambda df, c: int(round(df[c].mean(), 0)), 'pub_rec_bankruptcies'), 
} 

flag_mapper:Dict[str, Tuple[TransformerFunc, str]] = { 
	# ! Flag missing date values instead of imputations 
	'last_pymnt_d': (lambda df, c: pd.notnull(df[c]), 'last_pymnt_d_flag'), # flag null values 
	'last_credit_pull_d': (lambda df, c: pd.notnull(df[c]), 'last_credit_pull_d_flag'), # flag null values 
} 

temporary_variables = ['last_pymnt_d_flag', 'last_credit_pull_d_flag'] 
varname_to_remove = [
  # ! utiliser addr_state
  'zip_code', 
  # ! C'est valeurs semblent etre obtenues apres le pret 
  'last_pymnt_d', 'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv', 
  'total_pymnt', 'total_pymnt_inv', 'total_rec_late_fee', 
  'recoveries', 'collection_recovery_fee', 
] 

### Execution des transformations

In [ ]:

df_transformed = df_data.copy()


# ! Transformations de notre variables cible. 
# ! If loan_status == 'Fully Paid' then y = 1 else y = 0 
df_transformed['y'] = [ 1 if v == 'Fully Paid' else 0 for v in df_transformed['loan_status'].values ] 


# ! Transformations 
for varname, (func, outvarname) in transform_mapper.items(): 
	outvarname = outvarname or varname 
	df_transformed[outvarname] = func(df_transformed, varname) 

# ! Imputations 
for varname, (func, outvarname) in imputations_mapper.items(): 
	outvarname = outvarname or varname 
	df_transformed[outvarname] = df_transformed[varname].fillna(func(df_transformed, varname)) 

# ? Not necessary if we drop date columns 
# ! Flagging
for varname, (func, outvarname) in flag_mapper.items(): 
	outvarname = outvarname or varname 
	df_transformed[outvarname] = func(df_transformed, varname) 


# ! Drop flagged observations 
varname_flag = ['last_pymnt_d_flag', 'last_credit_pull_d_flag'] #  
to_keep = df_transformed[varname_flag].all(axis=1) 
df_transformed['to_keep'] = to_keep
df_rejected = df_transformed[~to_keep]  # ? Keep rejected observations for documentation purposes 
df_transformed = df_transformed[to_keep] 

# ! Remove temporary variables 
df_transformed = df_transformed.drop(columns=['to_keep', *varname_id, *varname_cardinality_1, *temporary_variables, *varname_to_remove]) 

# ! Ordonnancement des colonnes 
ordered = ['y', *[ c for c in df_transformed.columns if c != 'y']] 
df_transformed = df_transformed[ordered] 
df_transformed 


,y,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,total_rec_prncp,total_rec_int,last_credit_pull_d,pub_rec_bankruptcies
0,1,5000,5000,4975.0,36 months,10.65,162.87,B,B2,10+ years,...,1,3,0,13648,83.7,9,5000.00,863.16,2017-01-01,0.0
1,0,2500,2500,2500.0,60 months,15.27,59.83,C,C4,< 1 year,...,5,3,0,1687,9.4,4,456.46,435.17,2016-10-01,0.0
2,1,2400,2400,2400.0,36 months,15.96,84.33,C,C5,10+ years,...,2,2,0,2956,98.5,10,2400.00,605.67,2017-01-01,0.0
3,1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,10+ years,...,1,10,0,5598,21.0,37,10000.00,2214.92,2016-04-01,0.0
4,1,3000,3000,3000.0,60 months,12.69,67.79,B,B5,1 year,...,0,15,0,27783,53.9,38,3000.00,1066.91,2017-01-01,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39781,1,2500,2500,1075.0,36 months,8.07,78.42,A,A4,4 years,...,0,13,0,7274,13.1,40,2500.00,322.97,2010-06-01,0.0
39782,1,8500,8500,875.0,36 months,10.28,275.38,C,C1,3 years,...,1,6,0,8847,26.9,9,8500.00,1413.49,2010-07-01,0.0
39783,1,5000,5000,1325.0,36 months,8.07,156.84,A,A4,< 1 year,...,0,11,0,9698,19.4,20,5000.00,272.16,2007-06-01,0.0
39784,1,5000,5000,650.0,36 months,7.43,155.38,A,A2,< 1 year,...,0,17,0,85607,0.7,26,5000.00,174.20,2007-06-01,0.0


## Dummies here ?

## Standardisation 

## Distributional summary (Apres transformation)

In [73]:
summary = distributional_summary(df_transformed) 
summary 

,y,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,total_rec_prncp,total_rec_int,last_credit_pull_d,pub_rec_bankruptcies
type,numerical,numerical,numerical,numerical,string,numerical,numerical,string,string,string,...,numerical,numerical,numerical,numerical,numerical,numerical,numerical,numerical,date,numerical
dtype,int64,int64,int64,float64,object,float64,float64,object,object,object,...,int64,int64,int64,int64,float64,int64,float64,float64,datetime64[ns],float64
N,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713,...,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713
count,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713,...,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713
missing_p,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cardinality,2,885,1042,8215,2,371,15387,7,35,11,...,9,40,5,21726,1089,82,6919,35044,114,3
unique_p,0.0,0.00491,0.004432,0.179589,0.0,0.000428,0.228464,0.0,0.0,0.0,...,0.0,0.000126,0.0,0.305114,0.002241,0.000327,0.150706,0.810994,0.000151,0.0
most_freq,1,10000,10000,5000.0,36 months,10.99,311.11,B,B3,10+ years,...,0,7,0,0,0.0,16,10000.0,1196.57,2017-01-01 00:00:00,0.0
least_freq,0,15325,1125,9994.464812,60 months,16.71,255.43,G,G5,9 years,...,8,39,4,85607,34.89,77,721.02,3268.68,2008-06-01 00:00:00,2.0
mean,0.858837,11237.001737,10963.862589,10415.505431,NaN,12.024434,324.870992,NaN,NaN,NaN,...,0.868305,9.297938,0.05507,13404.427316,48.857432,22.10173,9873.357015,2280.492476,NaN,0.042455


In [74]:
v_completeness = degree_completeness(df_transformed) 
h_completeness = degree_completeness(df_transformed, axis=1) 

print(f"Completude 100%? {all(v_completeness['completeness']==1)}")
v_completeness.sort_values(by='completeness') 



Completude 100%? True


,completeness
y,1.0
loan_amnt,1.0
funded_amnt,1.0
funded_amnt_inv,1.0
term,1.0
int_rate,1.0
installment,1.0
grade,1.0
sub_grade,1.0
emp_length,1.0


## Graph distribution (Apres transformation)